# Stretch: more questions for the listening log, worked solutions

The answers to the stretch notebook. Try each one before looking.

## Setup

In [ ]:
import csv
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

PLAYS = "data/clean/plays.csv"
ARTISTS = "data/clean/artists.csv"


def load_rows(path):
    """Return the CSV at path as a list of dictionaries, one per row."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def top(counts, n=5):
    """Return the n highest (key, value) pairs, biggest first."""
    return sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:n]


rows = load_rows(PLAYS)
print(f"loaded {len(rows)} rows")

## Stretch 1: plays per month

In [ ]:
date = "2024-10-17"
print(date[:4])      # 2024
print(date[:7])      # 2024-10
print(date[5:7])     # 10

In [ ]:
def plays_per_month(rows):
    """Return a dictionary of 'YYYY-MM' -> number of plays in that month."""
    counts = {}
    for row in rows:
        month = row["played_at"][:7]
        counts[month] = counts.get(month, 0) + 1
    return counts


months = plays_per_month(rows)

assert len(months) == 24                       # two full years
assert sum(months.values()) == len(rows)       # every play counted once

for month, number in sorted(months.items()):
    print(f"{month}  {number:>4}")

`sorted(months.items())` with no `key` sorts the pairs by their first item,
the month, so they come out in date order. No `lambda` needed this time.

In [ ]:
print("busiest: ", top(months, 3))
print("quietest:", top(months, 24)[-3:])

October 2024 is the busiest month, at 127 plays. The quietest three are all
in summer 2025: August with 57, June and July with 59 each. Scroll back up
and 2024 has the same summer dip: July 61, August 66, June 74. Both years,
the same shape: quieter in summer, busier again in autumn and winter. It is
the kind of pattern you only see once the counts sit side by side.

**Why does sorting text give date order?** Because every month is written
with two digits. `"2024-02"` comes before `"2024-10"` because `"0"` comes
before `"1"`. Had the dates been written `"2024-2"`, the text `"2024-2"`
would sort *after* `"2024-10"`, because `"2"` comes after `"1"`. This is
the reason `YYYY-MM-DD` is the date format data people prefer: sorting it as
text and sorting it as dates give the same answer.

## Stretch 2: each artist's tracks

In [ ]:
def tracks_by_artist(rows):
    """Return a dictionary of artist -> list of their distinct track names."""
    tracks = {}
    for row in rows:
        artist = row["artist_name"]
        track = row["track_name"]
        if artist not in tracks:
            tracks[artist] = []
        if track not in tracks[artist]:
            tracks[artist].append(track)
    return tracks


catalogue = tracks_by_artist(rows)

assert len(catalogue["Glass Tram"]) == 4
assert len(catalogue["Odessa Brass"]) == 1
assert catalogue["Odessa Brass"] == ["Golden Avenue"]

print(sorted(catalogue["Glass Tram"]))

Two checks inside the loop, and they do different jobs:

* `if artist not in tracks` makes the empty list the first time an artist
  appears. Without it, `tracks[artist].append(...)` is a `KeyError`.
* `if track not in tracks[artist]` stops the same track being added once
  for every time it was played.

Now count the catalogues.

In [ ]:
sizes = {}
for artist, names in catalogue.items():
    sizes[artist] = len(names)

for artist, size in top(sizes, 6):
    print(f"  {artist:24} {size}")

The biggest catalogue is 9 tracks, and five artists share it: DJ Kompas,
Fjord & Flint, Kito Ndungu, Northern Signal and Static Meadow. So printing
six rows was the right call; with five you would not know it was a tie.

**The trap.** "How many different tracks are in the log?" has two answers.

In [ ]:
names = set(row["track_name"] for row in rows)
print(len(names), "different track names")

pairs = 0
for artist in catalogue:
    pairs += len(catalogue[artist])
print(pairs, "different tracks")

print(f"'Paper Signal' is by: {[a for a in catalogue if 'Paper Signal' in catalogue[a]]}")

168 different names, but 229 different tracks, because some names are used
by more than one artist: three different artists have a track called
*Paper Signal*. A track is only identified by its name **and** its artist
together. Real music services have the same problem, which is why they give
every track an ID number instead.

## Stretch 3: a second file, and a lookup

In [ ]:
artist_rows = load_rows(ARTISTS)
print(len(artist_rows), "artists")
print(artist_rows[0])

In [ ]:
def index_by(rows, column):
    """Return a dictionary of column value -> the whole row, for fast lookup."""
    index = {}
    for row in rows:
        index[row[column]] = row
    return index


artists = index_by(artist_rows, "artist_name")

assert artists["Glass Tram"]["country"] == "Germany"
print(artists["Glass Tram"])

Now plays per decade the artist was formed in.

In [ ]:
def decade(year_text):
    """Return a year like '2014' as a decade label like '2010s', or 'unknown' if blank."""
    if year_text == "":
        return "unknown"
    return f"{int(year_text) // 10 * 10}s"


assert decade("2014") == "2010s"
assert decade("2020") == "2020s"
assert decade("1996") == "1990s"
assert decade("") == "unknown"


def plays_by_decade(rows, artists):
    """Return a dictionary of decade formed -> number of plays."""
    counts = {}
    for row in rows:
        formed = artists[row["artist_name"]]["formed_year"]
        label = decade(formed)
        counts[label] = counts.get(label, 0) + 1
    return counts


by_decade = plays_by_decade(rows, artists)
assert sum(by_decade.values()) == len(rows)

for label, number in sorted(by_decade.items()):
    print(f"  {label:8} {number:>5}")

Artists formed in the 2010s get half of all the plays: 1,087 of 2,183.
53 plays belong to artists whose year is missing.

In [ ]:
missing = []
for row in artist_rows:
    if row["formed_year"] == "":
        missing.append(row["artist_name"])
print("no formed_year:", missing)

**What to do with the blanks.** `int("")` crashes with a `ValueError`, so
you have to decide something. Three options:

1. **skip those rows.** The decade counts would add up to 2,130 instead of
   2,183, and anyone checking the total would think plays had gone missing.
2. **pretend a year.** Then the 53 plays land in a decade they may not
   belong to, and nobody can tell afterwards. That is the worst option.
3. **give them their own label.** The totals still add up, and the reader
   can see exactly how much is unknown and judge whether it matters.

Option 3 is nearly always the honest one. 53 out of 2,183 is small enough
not to change the story here; if it had been 1,000, the story would be "we
do not know". Session 6 has the SQL version of this: `IS NULL`.

**Why a dictionary for the lookup?** Without `artists`, every one of the
2,183 rows would loop over all 40 artists to find its match: 87,320 steps.
With the dictionary, Python jumps straight to the right one. Indexing by a
name once, then looking up many times, is what a database does with a
`JOIN` in session 7.

The lookup also works the other way round. Who is in the artists file but
was never played?

In [ ]:
played = set(row["artist_name"] for row in rows)

never = []
for name in artists:
    if name not in played:
        never.append(name)

assert len(played) == 38
print("never played:", never)

Iron Fernway and Gravel Choir: 40 artists in the file, 38 in the log.
Neither file is wrong. They just do not cover the same things, and you only
find out by checking.

## Stretch 4: several numbers per artist, in one pass

In [ ]:
def artist_stats(rows):
    """Return a dictionary of artist -> {"plays", "minutes", "skipped"} totals."""
    stats = {}
    for row in rows:
        artist = row["artist_name"]
        if artist not in stats:
            stats[artist] = {"plays": 0, "minutes": 0.0, "skipped": 0}
        stats[artist]["plays"] += 1
        stats[artist]["minutes"] += float(row["minutes_played"])
        if row["skipped"] == "1":
            stats[artist]["skipped"] += 1
    return stats


stats = artist_stats(rows)

assert len(stats) == 38
assert stats["Odessa Brass"]["plays"] == 1
assert stats["Sara Lindqvist"]["plays"] == 242
assert stats["Glass Tram"]["skipped"] == 30
assert round(stats["Glass Tram"]["minutes"], 2) == 1175.07

glass = stats["Glass Tram"]
print(f"Glass Tram: {glass['plays']} plays, {glass['minutes']:.0f} minutes, "
      f"{glass['skipped']} skipped")

A dictionary whose values are dictionaries. Read
`stats[artist]["plays"]` from left to right: first find the artist, then
find that artist's play count.

The homework keeps two dictionaries side by side; this keeps one, with a
small row inside for each artist. Same information, different shape. Adding
a fourth number now means one more key, not one more dictionary.

In [ ]:
def skip_rates(stats, min_plays=1):
    """Return artist -> percentage of plays skipped, for artists with at least min_plays."""
    rates = {}
    for artist, s in stats.items():
        if s["plays"] >= min_plays:
            rates[artist] = 100 * s["skipped"] / s["plays"]
    return rates


print("every artist:")
for artist, rate in top(skip_rates(stats), 3):
    print(f"  {artist:24} {rate:5.1f}%  ({stats[artist]['plays']} plays)")

print("\nat least 30 plays:")
for artist, rate in top(skip_rates(stats, min_plays=30), 3):
    print(f"  {artist:24} {rate:5.1f}%  ({stats[artist]['plays']} plays)")

**Why the minimum?** Without it, the most-skipped artist is Circuit
Sparrow at 28.6%. That is 2 skips out of 7 plays. One fewer skip and the
rate halves. Kanto Kids, in second place, is 2 out of 8.

With at least 30 plays, the answer is Mireille Basset: 8 skips out of 40,
20.0%. Still not a huge number, but at least one play more or less does not
swing it wildly. For comparison, the whole log has a skip rate of 11.8%.

And the bottom of the unfiltered list is just as misleading: Odessa Brass
has a perfect 0% skip rate, from a single play.

**Why 30?** There is no correct number. 30 is a judgement call, and
that is fine, as long as you say what you chose. A ranking that hides its
cut-off is how small numbers end up in headlines.

In [ ]:
assert stats["Circuit Sparrow"]["skipped"] == 2
assert stats["Mireille Basset"]["skipped"] == 8

## Stretch 5: a table of your own, written to a file

In [ ]:
def genre_by_year(rows):
    """Return a dictionary of genre -> {year -> number of plays}."""
    table = {}
    for row in rows:
        genre = row["genre"]
        year = row["played_at"][:4]
        if genre not in table:
            table[genre] = {}
        table[genre][year] = table[genre].get(year, 0) + 1
    return table


table = genre_by_year(rows)
assert table["Jazz"] == {"2024": 72, "2025": 82}
print(table["Folk"])

In [ ]:
def summary_rows(table):
    """Return a list of row dictionaries: genre, 2024, 2025, change."""
    result = []
    for genre in sorted(table):
        counts = table[genre]
        result.append({
            "genre": genre,
            "2024": counts["2024"],
            "2025": counts["2025"],
            "change": counts["2025"] - counts["2024"],
        })
    return result


summary = summary_rows(table)

os.makedirs("output", exist_ok=True)
with open("output/genre_by_year.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["genre", "2024", "2025", "change"])
    writer.writeheader()
    writer.writerows(summary)

print("wrote output/genre_by_year.csv")
for row in summary:
    print(f"  {row['genre']:12} {row['2024']:>5} {row['2025']:>5} {row['change']:>+5}")

Open the file in VS Code and check it: 8 rows plus a header.

The rows are ones you built yourself, so you choose the columns. That is the
difference from the filter homework, where every row came out of the
original file and the columns came along for free.

The story in the table: plays went down from 1,122 in 2024 to 1,061 in
2025. Folk fell most, by 34. Only Afrobeat (up 14) and Jazz (up 10) grew.

`{row['change']:>+5}` puts a `+` in front of positive numbers, so rises and
falls are easy to tell apart.

This shape, one row per genre and one column per year, is called a **pivot
table**. Spreadsheet people make them all day, and in session 8 pandas makes
one in a single line. Now you know what that line is doing.

## Stretch 6: read it back, and check it

In [ ]:
back = load_rows("output/genre_by_year.csv")

assert len(back) == len(summary)
print(back[0])

Everything came back as **text**. The file was written from numbers, and the
numbers did not survive the trip; a CSV only stores characters.

That breaks things quietly. Ask "which genre had the most plays in 2025?"
without converting:

In [ ]:
wrong = {}
right = {}
for row in back:
    wrong[row["genre"]] = row["2025"]
    right[row["genre"]] = int(row["2025"])

print("as text:   ", top(wrong, 1))
print("as numbers:", top(right, 1))

As text, Afrobeat with `"97"` wins, because text compares one character at a
time and `"9"` beats `"2"`. As numbers, Electronic with 268 wins. No error,
just a confident wrong answer.

Now the check that makes this a check: the file should agree with the raw
data it came from.

In [ ]:
total_2024 = 0
total_2025 = 0
for row in back:
    total_2024 += int(row["2024"])
    total_2025 += int(row["2025"])

plays_2024 = 0
for row in rows:
    if row["played_at"].startswith("2024"):
        plays_2024 += 1

assert total_2024 == plays_2024 == 1122
assert total_2024 + total_2025 == len(rows)
print(f"file says {total_2024} + {total_2025} = {total_2024 + total_2025}, the log has {len(rows)}")

Two independent routes to the same number, and they agree. Writing a file
and then reading it back to check the totals is a habit that catches real
bugs: a filter that dropped rows, a `"w"` that overwrote the wrong file, a
column that went missing.

In session 10's pipeline, this exact kind of check is what tells you whether
a run worked.